# VFA-1 — Tabular Q-Learning Experiment

Trains a `TabularQAgent` on a 10×10 `FogGridWorld` for 500 episodes, then animates
the final greedy episode step-by-step as a side-by-side pair:
**left** = agent view (fog on), **right** = full map (fog off).

This notebook covers:
1. Training loop with progress logging
2. Learning curve + Q-table growth
3. Final path — live animation, fog view vs. full map

In [31]:
import sys
from pathlib import Path

# Locate vfa1_navigating_fog/ regardless of where Jupyter was launched
root = Path.cwd()
while not (root / 'src' / 'environment').exists() and root != root.parent:
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from src.environment.grid_world import FogGridWorld
from src.agents.tabular_q import TabularQAgent

%matplotlib inline
plt.rcParams['figure.dpi'] = 110

## 1. Train

Hyperparameters follow the VFA-1 design spec.  Epsilon decays multiplicatively
each episode: `ε ← max(ε_end, ε × ε_decay)`.

A new random map is generated every `reset()` call, so the agent cannot memorise
a fixed layout — generalisation is required for good performance.

In [ ]:
N_TRAIN = 5000   # training episodes
GRID    = 10    # grid side length
SMOOTH  = 20    # rolling-average window for the learning curve

env = FogGridWorld(grid_size=GRID)
agent = TabularQAgent(
    n_actions     = env.action_space.n,
    alpha         = 0.1,
    gamma         = 0.99,
    epsilon_start = 1.0,
    epsilon_end   = 0.05,
    epsilon_decay = 0.995,
)

episode_rewards: list[float] = []
qtable_sizes:   list[int]   = []

for ep in range(1, N_TRAIN + 1):
    obs, _ = env.reset()
    ep_reward = 0.0
    done = False

    while not done:
        action = agent.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        agent.update(obs, action, reward, next_obs, terminated)
        obs = next_obs
        ep_reward += reward
        done = terminated or truncated

    agent.decay_epsilon()
    episode_rewards.append(ep_reward)
    qtable_sizes.append(agent.q_table_size)

    if ep % 100 == 0:
        avg = float(np.mean(episode_rewards[-50:]))
        print(
            f'ep {ep:>4}  '
            f'avg_reward(50) {avg:+.3f}  '
            f'eps {agent.epsilon:.3f}  '
            f'Q-entries {agent.q_table_size:,}'
        )

ep  100  avg_reward(50) -1.264  eps 0.606  Q-entries 18,082
ep  200  avg_reward(50) -1.164  eps 0.367  Q-entries 35,028
ep  300  avg_reward(50) -1.472  eps 0.222  Q-entries 53,701
ep  400  avg_reward(50) -1.541  eps 0.135  Q-entries 71,782


### Learning curve

**Top panel** — episode rewards (raw in light blue, 20-episode rolling average in solid blue).
The dashed line marks reward = 0; positive values indicate net-profitable episodes.

**Bottom panel** — Q-table size (unique state–action pairs seen).  Growth slows once the agent
stops visiting genuinely new observations — a sign of partial convergence on this grid size.

In [ ]:
fig, (ax_r, ax_q) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

episodes = np.arange(1, N_TRAIN + 1)

# ── Reward panel ─────────────────────────────────────────────────────────────
ax_r.plot(episodes, episode_rewards,
          alpha=0.25, color='steelblue', linewidth=0.8, label='episode reward')
kernel   = np.ones(SMOOTH) / SMOOTH
smoothed = np.convolve(episode_rewards, kernel, mode='valid')
ax_r.plot(episodes[SMOOTH - 1:], smoothed,
          color='steelblue', linewidth=2.0, label=f'{SMOOTH}-ep rolling avg')
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax_r.set_ylabel('Episode reward')
ax_r.legend(loc='lower right', fontsize=9)
ax_r.set_title(
    f'Tabular Q-Learning — {GRID}×{GRID} grid, {N_TRAIN} episodes',
    fontsize=12,
)

# ── Q-table size panel ────────────────────────────────────────────────────────
ax_q.plot(episodes, qtable_sizes, color='darkorange', linewidth=1.5)
ax_q.set_ylabel('Q-table entries')
ax_q.set_xlabel('Episode')
ax_q.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)

plt.tight_layout()
plt.show()

print(f'Final Q-table size: {agent.q_table_size:,} entries')
print(f'Final epsilon:      {agent.epsilon:.4f}')
print(f'Last-50 avg reward: {float(np.mean(episode_rewards[-50:])):+.4f}')

## 2. Final greedy episode — live animation

One episode is run with `greedy=True` (pure exploitation, no exploration).
Each step is rendered as it happens using `clear_output`, so you see the agent
move through the map in real time.

* **Left** — what the agent actually observes: cells outside the visible radius show as FOG.
* **Right** — ground truth: the complete map with all cell types revealed.

Adjust `DELAY` to control playback speed.

In [ ]:
import time
from IPython.display import clear_output

DEMO_SEED = 42
DELAY     = 0.20   # seconds between frames; lower = faster playback

# Pure exploitation: epsilon=0 means select_action always picks argmax Q
agent.epsilon = 0.0

env_vis = FogGridWorld(grid_size=GRID)
obs, _  = env_vis.reset(seed=DEMO_SEED)

ep_reward   = 0.0
last_reward = 0.0
done        = False

while not done:
    clear_output(wait=True)

    fig, (ax_fog, ax_full) = plt.subplots(1, 2, figsize=(12, 5))
    env_vis.render_frame(ax_fog,  fog=True)
    env_vis.render_frame(ax_full, fog=False)
    ax_fog.set_title('Agent view  (fog on)',  fontsize=11)
    ax_full.set_title('Full map  (fog off)', fontsize=11)
    fig.suptitle(
        f'Exploitation episode  ·  cumulative reward {ep_reward:+.3f}',
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    action = agent.select_action(obs)   # epsilon=0 → always greedy
    obs, last_reward, terminated, truncated, info = env_vis.step(action)
    ep_reward += last_reward
    done = terminated or truncated
    time.sleep(DELAY)

# ── Terminal frame ────────────────────────────────────────────────────────────
clear_output(wait=True)

fig, (ax_fog, ax_full) = plt.subplots(1, 2, figsize=(12, 5))
env_vis.render_frame(ax_fog,  fog=True)
env_vis.render_frame(ax_full, fog=False)
ax_fog.set_title('Agent view  (fog on)',  fontsize=11)
ax_full.set_title('Full map  (fog off)', fontsize=11)

if terminated and last_reward >= 0.9:
    result = 'GOAL reached'
elif terminated and last_reward <= -0.9:
    result = 'TRAP entered'
elif terminated:
    result = 'energy depleted'
else:
    result = f'truncated at {info["steps"]} steps'

fig.suptitle(
    f'Exploitation episode  ·  {info["steps"]} steps  ·  '
    f'reward {ep_reward:+.3f}  ·  {result}',
    fontsize=12,
)
plt.tight_layout()
plt.show()

## 3. Fixed-map baseline — proof of convergence

The random-map setting above prevents tabular Q-learning from converging: every
episode the layout changes, so Q-values learned for one map are noise for the
next.

Here we test the **same agent on a fixed 8×8 map** (`fixed_map=True`).  The
topology never changes between episodes — only the agent's starting position is
re-sampled.  With a consistent state → value mapping, the Q-table should
converge and average reward should rise toward **+1.0**.

This experiment serves as the **proof of principle**: tabular Q-learning *can*
solve a small grid when maps are fixed.  The contrast with the random-map curves
above isolates the exact failure mode that Linear FA must overcome.

In [ ]:
N_FIXED   = 2000   # training episodes
GRID_FIXED = 10    # small grid for fast convergence
SMOOTH_FIXED = 50  # rolling-average window

env_fixed = FogGridWorld(grid_size=GRID_FIXED, fixed_map=True)
agent_fixed = TabularQAgent(
    n_actions     = env_fixed.action_space.n,
    alpha         = 0.1,
    gamma         = 0.99,
    epsilon_start = 1.0,
    epsilon_end   = 0.05,
    epsilon_decay = 0.995,
)

fixed_rewards: list[float] = []
fixed_qtable:  list[int]   = []

for ep in range(1, N_FIXED + 1):
    obs, _ = env_fixed.reset()
    ep_reward = 0.0
    done = False

    while not done:
        action = agent_fixed.select_action(obs)
        next_obs, reward, terminated, truncated, _ = env_fixed.step(action)
        agent_fixed.update(obs, action, reward, next_obs, terminated)
        obs = next_obs
        ep_reward += reward
        done = terminated or truncated

    agent_fixed.decay_epsilon()
    fixed_rewards.append(ep_reward)
    fixed_qtable.append(agent_fixed.q_table_size)

    if ep % 200 == 0:
        avg = float(np.mean(fixed_rewards[-100:]))
        print(
            f'ep {ep:>4}  '
            f'avg_reward(100) {avg:+.3f}  '
            f'eps {agent_fixed.epsilon:.3f}  '
            f'Q-entries {agent_fixed.q_table_size:,}'
        )

In [ ]:
fig, (ax_r, ax_q) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

episodes_fixed = np.arange(1, N_FIXED + 1)

# ── Reward panel ──────────────────────────────────────────────────────────────
ax_r.plot(episodes_fixed, fixed_rewards,
          alpha=0.2, color='seagreen', linewidth=0.8, label='episode reward')
kernel_f   = np.ones(SMOOTH_FIXED) / SMOOTH_FIXED
smoothed_f = np.convolve(fixed_rewards, kernel_f, mode='valid')
ax_r.plot(episodes_fixed[SMOOTH_FIXED - 1:], smoothed_f,
          color='seagreen', linewidth=2.0, label=f'{SMOOTH_FIXED}-ep rolling avg')
ax_r.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax_r.axhline(1, color='gold', linestyle=':', linewidth=1.0, label='goal reward (+1)')
ax_r.set_ylabel('Episode reward')
ax_r.legend(loc='lower right', fontsize=9)
ax_r.set_title(
    f'Tabular Q-Learning — {GRID_FIXED}×{GRID_FIXED} FIXED map, {N_FIXED} episodes\n'
    f'(fixed_map=True: same layout every episode)',
    fontsize=12,
)

# ── Q-table size panel ────────────────────────────────────────────────────────
ax_q.plot(episodes_fixed, fixed_qtable, color='darkorange', linewidth=1.5)
ax_q.set_ylabel('Q-table entries')
ax_q.set_xlabel('Episode')
ax_q.yaxis.set_major_formatter(
    mticker.FuncFormatter(lambda x, _: f'{int(x):,}')
)

plt.tight_layout()
plt.show()

last100_avg = float(np.mean(fixed_rewards[-100:]))
print(f'Final Q-table size:       {agent_fixed.q_table_size:,} entries')
print(f'Final epsilon:            {agent_fixed.epsilon:.4f}')
print(f'Last-100 avg reward:      {last100_avg:+.4f}')
print(f'Converged (avg > 0.5)?    {"YES" if last100_avg > 0.5 else "NO"}')

### Final greedy episode on the fixed map

Same agent, same map it was trained on — epsilon forced to 0 (pure exploitation).
Left = agent view with fog; right = full map revealed.

In [ ]:
DELAY_FIXED = 0.20  # seconds between frames

# Pure exploitation on the same map the agent was trained on
agent_fixed.epsilon = 0.0

obs, _ = env_fixed.reset()   # fixed map restored; random start position
ep_reward   = 0.0
last_reward = 0.0
done        = False

while not done:
    clear_output(wait=True)

    fig, (ax_fog, ax_full) = plt.subplots(1, 2, figsize=(12, 5))
    env_fixed.render_frame(ax_fog,  fog=True)
    env_fixed.render_frame(ax_full, fog=False)
    ax_fog.set_title('Agent view  (fog on)',  fontsize=11)
    ax_full.set_title('Full map  (fog off)', fontsize=11)
    fig.suptitle(
        f'Fixed-map exploitation  ·  cumulative reward {ep_reward:+.3f}',
        fontsize=12,
    )
    plt.tight_layout()
    plt.show()

    action = agent_fixed.select_action(obs)
    obs, last_reward, terminated, truncated, info = env_fixed.step(action)
    ep_reward += last_reward
    done = terminated or truncated
    time.sleep(DELAY_FIXED)

# ── Terminal frame ────────────────────────────────────────────────────────────
clear_output(wait=True)

fig, (ax_fog, ax_full) = plt.subplots(1, 2, figsize=(12, 5))
env_fixed.render_frame(ax_fog,  fog=True)
env_fixed.render_frame(ax_full, fog=False)
ax_fog.set_title('Agent view  (fog on)',  fontsize=11)
ax_full.set_title('Full map  (fog off)', fontsize=11)

if terminated and last_reward >= 0.9:
    result = 'GOAL reached'
elif terminated and last_reward <= -0.9:
    result = 'TRAP entered'
elif terminated:
    result = 'energy depleted'
else:
    result = f'truncated at {info["steps"]} steps'

fig.suptitle(
    f'Fixed-map exploitation  ·  {info["steps"]} steps  ·  '
    f'reward {ep_reward:+.3f}  ·  {result}',
    fontsize=12,
)
plt.tight_layout()
plt.show()